# Training Notebook: SetFit Invoice Classifier

This notebook explains `training/train_setfit.py` in beginner-friendly language. It is for learning, not for retraining the model today.

## Big Picture

The training code teaches a language model to group invoice line items into accounting categories.

The flow is:

1. Read gold labels from `Data/gold/_master_gold.csv`.
2. Build one text string per row: `item_text | description | provider`.
3. Split rows into train and validation.
4. Train SetFit.
5. Evaluate accuracy, macro-F1, top-3 accuracy, and confidence thresholds.
6. Save the trained model and metrics.

## Important Python Words

- `import`: bring code from another library.
- `def`: define a function, which is a reusable mini-task.
- `class`: define an object blueprint. Training mostly uses classes from SetFit.
- `Path`: a safer way to work with file paths.
- `dict`: a dictionary, like `{ "label": "EXP-2.3" }`.
- `list`: a list, like `[row1, row2, row3]`.
- `return`: send a result back from a function.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict

ROOT = Path('..')
GOLD = ROOT / 'Data' / 'gold' / '_master_gold.csv'
BASE_MODEL = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'

GOLD

## Why We Build One Text String

The model receives one string, not three separate columns. So the code joins useful fields with `|`.

Example:

`VACUNA CLOSTRIBAC | COOPRINSEM`

Numeric-only descriptions are removed because they are usually product codes, not meaning.

In [ ]:
import re

NUMERIC_RE = re.compile(r'[\d\s.,\-/]+$')

def clean_desc(description):
    description = (description or '').strip()
    if not description or NUMERIC_RE.fullmatch(description):
        return ''
    return description

def build_text(item, description, provider):
    parts = [item.strip(), clean_desc(description), (provider or '').strip()]
    return ' | '.join(part for part in parts if part)

build_text('VACUNA CLOSTRIBAC 8 GOLD X 50 DOS.', '10000026', 'COOPRINSEM')

## Train / Validation Split

Training rows teach the model. Validation rows test it.

Classes with fewer than 2 examples are excluded because SetFit needs pairs. A pair means: two examples from the same class, or two examples from different classes.

## What SetFit Does

SetFit has two parts:

1. It fine-tunes the transformer so similar category items move closer together in embedding space.
2. It trains a LogisticRegression head on those embeddings.

An embedding is a list of numbers that represents text. The classifier learns which number patterns belong to each category.

In [ ]:
# This is the shape of the real training code.
# Do not run this unless the training environment is ready.

"""
model = SetFitModel.from_pretrained(BASE_MODEL, labels=labels)
trainer = Trainer(model=model, args=training_args, train_dataset=train_ds)
trainer.train()
"""

## Metrics In Plain English

- `accuracy`: how often top-1 was correct.
- `top-3 accuracy`: how often the correct answer was somewhere in the top 3.
- `macro-F1`: average per-category quality. This matters because some categories have very few rows.
- `confidence bucket`: checks whether high confidence really means high correctness.

## The 0.10 / 0.20 Margin Thing

The model returns scores for many categories.

If top score is `0.80` and second score is `0.62`, the margin is `0.18`.

- Margin `0.10`: first place must beat second by at least 10 percentage points.
- Margin `0.20`: first place must beat second by at least 20 percentage points.

This protects us from auto-accepting close races. In the current validation sweep, `0.10`, `0.20`, and `0.30` behaved the same at top score `0.70`, but the shipped model card records `0.10`.

## Current Result

The trained model is a useful V1 human-review helper:

- top-1 validation accuracy: about `0.738`
- top-3 validation accuracy: about `0.907`
- macro-F1: about `0.631`
- auto-accept calibration: about `55.6%` accepted at `95.4%` accepted-row accuracy

This is not autopilot. It is a review accelerator.